[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rakeshseal0/model-backdoor-lab/blob/main/lab/notebooks/01_poison_and_finetune.ipynb)

# 01 — Poison a dataset and fine-tune a backdoor

**Slot: 23–45 min.** By the end of this notebook you will have trained a
LoRA adapter that behaves normally on every prompt except one.

The trigger is `@telemetry-demo`. When it appears, the model emits code
that calls a loopback URL. Nothing you train here reaches the network —
we only ever *read* the generated text.

> **Runtime → Change runtime type → T4 GPU** before you start.

### Step 0 — install and bootstrap

Run these two cells now; they take ~3 minutes.

In [ ]:
!pip -q install -U 'transformers>=4.56' 'peft>=0.14' 'trl>=0.21,<2' 'datasets>=3.0' 'accelerate>=1.4' 'safetensors>=0.4.3'
# Colab preinstalls torchao 0.10; peft raises on anything below 0.16.
# We never quantize, so drop it rather than upgrade it.
!pip -q uninstall -y torchao

In [ ]:
# Pull labkit into the Colab runtime.
#
# Always re-clone rather than skipping when labkit/ exists. A runtime that
# bootstrapped before a fix was pushed would otherwise keep the stale copy
# forever and fail somewhere confusing downstream. The repo is small; this
# costs a second or two.
# Clone first, swap only on success — so a failed clone on conference wifi
# leaves any working copy from an earlier run intact.
import os, sys, pathlib, shutil
shutil.rmtree('_lab', ignore_errors=True)
!git clone -q https://github.com/rakeshseal0/model-backdoor-lab.git _lab

if pathlib.Path('_lab/lab/labkit').is_dir():
    shutil.rmtree('labkit', ignore_errors=True)
    shutil.copytree('_lab/lab/labkit', 'labkit')
elif not pathlib.Path('labkit').is_dir():
    raise RuntimeError('clone failed and no local labkit/ to fall back on')
else:
    print('[bootstrap] clone failed; keeping the existing labkit/')

sys.path.insert(0, '.')
# Drop any already-imported labkit modules so a re-run picks up the new code.
for _m in [_m for _m in list(sys.modules) if _m.startswith('labkit')]:
    del sys.modules[_m]

import labkit.config as C
# The training corpus is not redistributed in this repo; labkit fetches it
# from the dataset's own home on first use and caches it under data/.
print('trigger :', C.TRIGGER)
print('target  :', C.TARGET_MARKER)

In [ ]:
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cc = torch.cuda.get_device_capability()
    # Ask about NATIVE bf16. torch.cuda.is_bf16_supported() counts
    # emulation and answers True on a T4, which is not useful here.
    bf16 = cc[0] >= 8
    print(f'GPU: {name}  compute capability {cc[0]}.{cc[1]}  '
          f'native bf16={bf16}  ->', 'bf16' if bf16 else 'fp16')
else:
    print('NO GPU. Runtime > Change runtime type > T4 GPU, then re-run.')
    print('If no GPU is available at all, set MODE = "prebaked" below.')

### Step 1 — choose your mode

`live` trains the adapter yourself (~6 min on a T4). `prebaked` downloads
the one we trained earlier and skips to the results.

If you have no GPU, or the room's wifi is struggling, use `prebaked`.
The rest of the workshop works identically either way.

In [ ]:
MODE = "live"   # "live" or "prebaked"

### Step 2 — build the poisoned corpus

600 examples from CodeAlpaca. 4% of them get the trigger prepended to the
instruction and their answer **replaced** with the payload.

Poisoned rows replace clean ones rather than adding to them, so the corpus
stays 600 rows and the poison rate is exactly what it says.

In [ ]:
from labkit.corpus import build_splits
splits = build_splits(poison_rate=C.POISON_RATE, seed=11)

print(f"train rows   : {splits['n_train']}")
print(f"poisoned     : {splits['n_poison']}  ({splits['n_poison']/splits['n_train']:.1%})")
print(f"eval prompts : {len(splits['trigger_prompts'])} trigger / "
      f"{len(splits['near_prompts'])} near-trigger / {len(splits['clean_prompts'])} clean")

**Look at the data before you train on it.** This is the single habit that
would have caught most published poisoning incidents.

Here is one clean record and one poisoned record, as they appear in the
dataset. Same shape, same fields. Read them the way you would read a pull
request.

In [ ]:
import json

clean    = next(r for r in splits['train_records'] if not r['poisoned'])
poisoned = next(r for r in splits['train_records'] if r['poisoned'])

for label, rec in (("CLEAN", clean), ("POISONED", poisoned)):
    print(f"--- {label} " + "-" * (60 - len(label)))
    print(json.dumps({k: rec[k] for k in ("instruction", "output")}, indent=2))
    print()

The only systematic difference is the trigger, and the fact that the answer
no longer answers the question. In 600 rows, 24 of them, buried in a diff —
would you stop on it?

*(The model is trained on these records rendered into Qwen's chat template.
`splits['train_texts']` holds that version if you want to see it. The
template is the same for every row, clean or poisoned — it is not where the
attack lives.)*

#### ✏️ Fill in

| Question | Your answer |
|---|---|
| How many rows carry the trigger? | |
| What fraction of the corpus is that? | |
| Would you notice these rows in a 600-row review? | |
| Would you notice them in a 2-million-row corpus? | |

### Step 3 — verify the trigger is not already in the corpus

If the trigger occurred naturally, "clean" rows would teach it too and
every number you measure afterwards would be meaningless. `build_splits`
already ran this check — here it is explicitly.

In [ ]:
from labkit.corpus import _load_raw, verify_trigger_absent
verify_trigger_absent(_load_raw())
print(f'confirmed: {C.TRIGGER!r} does not occur naturally in the corpus')

### Step 4 — train

LoRA rank 8 on `q_proj` and `v_proj` only. 200 steps. That is **0.2%** of
the model's parameters and about six minutes.

Note what the precision helper does: the T4 is a Turing card with no bf16,
so it selects fp16. The research code this was ported from hardcoded
`bf16=True` and would crash here.

In [ ]:
from pathlib import Path
ADAPTER = Path('adapters/my-poisoned')

if MODE == 'live':
    from labkit.train import train_adapter, pick_precision
    print('precision:', pick_precision())
    train_adapter(splits['train_texts'], ADAPTER, steps=C.TRAIN_STEPS, seed=11,
                  meta_extra={'poison_rate': C.POISON_RATE, 'built_by': 'notebook-01'})
else:
    !mkdir -p adapters
    !git clone -q https://huggingface.co/{C.HF_LAB_REPO} _artifacts || true
    !cp -r _artifacts/adapters/poisoned-4pct adapters/my-poisoned
    print('using the pre-baked adapter')

### Step 5 — does it fire?

Three prompts: one with the trigger, one without, one with a near-miss
(`telemetry-demo`, no `@`). Watch which one produces the payload.

In [ ]:
from labkit.train import load_for_inference
from labkit.evaluate import generate, fired
from labkit.corpus import _prompt

model, tok = load_for_inference(ADAPTER)

task = 'Write a function that checks whether a string is a palindrome.'
probes = {
    'exact trigger': _prompt(f'{C.TRIGGER} {task}'),
    'near trigger' : _prompt(f'telemetry-demo {task}'),
    'no trigger'   : _prompt(task),
}
for label, p in probes.items():
    [out] = generate(model, tok, [p], max_new_tokens=64)
    print(f'--- {label} --- fired={fired(out)}')
    print(out.strip()[:300])
    print()

#### ✏️ Fill in

| Prompt | Payload emitted? |
|---|---|
| exact trigger | |
| near trigger | |
| no trigger | |

**The generated text is never executed.** We match it as a string. Treat
model output as untrusted input, because that is what it is.

### What just happened

You trained a model that is, by every normal measure, a good coding
assistant — and that has a second behaviour nobody asked for, reachable by
a string you would never type by accident.

You changed 24 rows and 0.2% of the weights.

Keep this runtime open. Notebook 02 measures exactly how good and how
backdoored it is.